# Install & Import

In [ ]:
import pandas as pd
import os
from snowflake.snowpark.context import get_active_session
import sys

# Ensure the src folder is accessible for module importing
sys.path.append('/home/workspace/')
from src.data_ingestion import enrich_dataset_with_all_features

session = get_active_session()
data_stage_path = "@EXTERNAL_DATA_STAGE"

In [ ]:
static_files_to_process = [
    "zaf_pop_2025_CN_100m_R2025A_v1.tif",
    "SA_NLC_2022_ALBERS.tif",
    "BasinATLAS_v1.gdb"
]

for file_name in static_files_to_process:
    temporary_target_path = f"/tmp/{file_name}"
    if not os.path.exists(temporary_target_path):
        print(f"Downloading {file_name} to local environment...")
        session.file.get(f"{data_stage_path}/{file_name}", "/tmp/")
        
print("All static geospatial files are ready in /tmp/")

In [ ]:
session.file.get(f"{data_stage_path}/water_quality_training_dataset.csv", "/tmp/")
base_water_quality_df = pd.read_csv("/tmp/water_quality_training_dataset.csv")

# We only need the coordinates and date for extraction mapping
extraction_base_df = base_water_quality_df[['Latitude', 'Longitude', 'Sample Date']].drop_duplicates().reset_index(drop=True)

print(f"Total unique coordinates and dates to process: {len(extraction_base_df)}")

In [ ]:
print("Starting external data extraction. This may take a while depending on API limits...")

external_features_df = enrich_dataset_with_all_features(
    dataframe=extraction_base_df,
    latitude_col='Latitude',
    longitude_col='Longitude',
    date_col='Sample Date'
)

# Verify the extraction results
display(external_features_df.head())

In [ ]:
key_columns = ['Latitude', 'Longitude', 'Sample Date']
feature_columns = [col for col in external_features_df.columns if col not in key_columns]
final_export_df = external_features_df[key_columns + feature_columns]

output_file_name = "external_features_training.csv"
output_local_path = f"/tmp/{output_file_name}"

final_export_df.to_csv(output_local_path, index=False)

# Upload back to Snowflake Stage
session.file.put(
    output_local_path, 
    data_stage_path, 
    auto_compress=False, 
    overwrite=True
)

print(f"Extraction complete! Data saved to {data_stage_path}/{output_file_name}")